In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.multi_sparse", False)

In [ ]:
REPO_ROOT = Path.cwd().resolve().parents[1]
CAL_DIR = REPO_ROOT / "results" / "calibration"

assert CAL_DIR.is_dir(), f"calibration results not found at {CAL_DIR}"

N_BOOT = 2000
SEED = 42

In [ ]:
MODEL_FAMILY = {
    "LogisticRegression": "Linear",
    "SVM": "Kernel",
    "RandomForest": "Tree ensemble",
    "LightGBM": "Tree ensemble",
    "XGBoost": "Tree ensemble",
    "MLP": "Neural network",
    "HybridLSTM": "Neural network",
    "Stacking": "Meta-ensemble",
}
FAMILY_ORDER = ["Linear", "Kernel", "Tree ensemble", "Neural network", "Meta-ensemble"]
MODEL_ORDER = sorted(MODEL_FAMILY, key=lambda m: FAMILY_ORDER.index(MODEL_FAMILY[m]))

MODEL_LABEL = {
    "LogisticRegression": "Logistic Regression",
    "SVM": "SVM",
    "RandomForest": "Random Forest",
    "LightGBM": "LightGBM",
    "XGBoost": "XGBoost",
    "MLP": "MLP",
    "HybridLSTM": "Hybrid LSTM",
    "Stacking": "Stacking",
}
MOMENT_ORDER = ["moment_1", "moment_2"]
MOMENT_LABEL = {"moment_1": "Shot release", "moment_2": "Rim arrival"}
BRANCH_LABEL = {True: "Balanced", False: "Class-weighted"}


MODEL_COLOR = dict(zip(MODEL_ORDER, [
    "#000000", "#E69F00", "#56B4E9", "#009E73",
    "#0072B2", "#D55E00", "#CC79A7", "#999999",
]))
BRANCH_LINESTYLE = {True: "--", False: "-"}
BRANCH_MARKER_FILL = {True: "none", False: "full"}

In [ ]:
TEXT_WIDTH_IN = 16 / 2.54
FIG_FULL = (TEXT_WIDTH_IN, TEXT_WIDTH_IN * 0.55)
FIG_HALF = (TEXT_WIDTH_IN / 2, TEXT_WIDTH_IN / 2 * 0.8)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "svg.fonttype": "path",
    "font.family": "serif",
    "font.serif": ["New Computer Modern", "Latin Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "legend.frameon": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#dddddd",
    "grid.linewidth": 0.5,
    "axes.axisbelow": True,
    "lines.linewidth": 1.2,
})

In [ ]:
summary = pd.read_csv(CAL_DIR / "calibration_summary.csv")
sweep = pd.read_csv(CAL_DIR / "calibration_threshold_sweep.csv")
preds = pd.read_parquet(CAL_DIR / "holdout_predictions.parquet")

def _prepare(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["name"] = pd.Categorical(df["name"], categories=MODEL_ORDER, ordered=True)
    df["moment"] = pd.Categorical(df["moment"], categories=MOMENT_ORDER, ordered=True)
    df["branch"] = df["balanced"].map(BRANCH_LABEL)
    return df

summary = _prepare(summary).sort_values(["moment", "balanced", "name"]).reset_index(drop=True)
sweep = _prepare(sweep)
preds = _prepare(preds)

unknown = set(pd.read_csv(CAL_DIR / "calibration_summary.csv")["name"]) - set(MODEL_ORDER)
assert not unknown, f"models missing from MODEL_ORDER: {unknown}"

print(f"{len(summary)} runs | {len(sweep)} sweep rows | {len(preds):,} prediction rows")
summary.groupby(["moment", "branch"], observed=True).size().unstack()

## **Calibration Results**

#### **Mean prediction vs real off rate**

In [ ]:
from IPython.display import display
from sklearn.calibration import calibration_curve

summary["model"] = summary["name"].map(MODEL_LABEL)
summary["gap"] = summary["cal_mean_pred"] - summary["true_off_rate"]
summary["baseline_brier"] = summary["true_off_rate"] * (1 - summary["true_off_rate"])
summary["brier_skill"] = 1 - summary["cal_brier"] / summary["baseline_brier"]


def plot_metric(col: str, label: str, ref: float | None = None, sharex: bool = False):
    fig, axes = plt.subplots(1, 2, figsize=FIG_FULL, sharey=True, sharex=sharex)
    for ax, moment in zip(axes, MOMENT_ORDER):
        d = summary[summary["moment"] == moment].sort_values("name")
        for balanced in (False, True):
            dd = d[d["balanced"] == balanced]
            ax.scatter(dd[col], dd["model"],  label=BRANCH_LABEL[balanced])
        if ref is not None:
            ax.axvline(ref, color="gray", ls=":", lw=1)
        ax.set_title(MOMENT_LABEL[moment])
        ax.set_xlabel(label)
    axes[0].invert_yaxis()
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.08))
    plt.tight_layout()
    plt.show()

In [ ]:
(summary.set_index(["moment", "model", "branch"])
        [["true_off_rate", "cal_mean_pred", "gap"]]
        .sort_values("gap", key=abs, ascending=False)
        .round(4))

In [ ]:
plot_metric("gap", "Mean predicted − actual OFF rate", ref=0, sharex=True)

#### **Reliability Plots**

In [ ]:
FAMILY_LS = {"Linear": "-", "Kernel": "-", "Tree ensemble": "--",
             "Neural network": "-.", "Meta-ensemble": ":"}

fig, axes = plt.subplots(2, 2, figsize=(TEXT_WIDTH_IN, TEXT_WIDTH_IN), sharex=True, sharey=True)
for i, moment in enumerate(MOMENT_ORDER):
    for j, balanced in enumerate([False, True]):
        ax = axes[i, j]
        ax.plot([0, 0.6], [0, 0.6], color="gray", ls=":", lw=1)
        d = preds[(preds["moment"] == moment) & (preds["balanced"] == balanced)]
        for name, g in d.groupby("name", observed=True):
            frac_pos, mean_pred = calibration_curve(g["y"], g["p_cal"], n_bins=10, strategy="quantile")
            ax.plot(mean_pred, frac_pos, color=MODEL_COLOR[name],
                    ls=FAMILY_LS[MODEL_FAMILY[name]], lw=0.9, alpha=0.85,
                    marker="o", ms=2, label=MODEL_LABEL[name])
        ax.set_title(f"{MOMENT_LABEL[moment]} — {BRANCH_LABEL[balanced]}")
        ax.set_xlim(0, 0.6)
        ax.set_ylim(0, 0.6)
        ax.set_aspect("equal")
for ax in axes[1]:
    ax.set_xlabel("Predicted probability")
for ax in axes[:, 0]:
    ax.set_ylabel("Actual OFF rate")

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.show()

#### **Raw vs Calibrated**

In [ ]:
(summary.set_index(["moment", "model", "branch"])
        [["raw_mean_pred", "cal_mean_pred", "raw_brier", "cal_brier", "raw_log_loss", "cal_log_loss"]]
        .assign(brier_change=lambda t: t["cal_brier"] - t["raw_brier"],
                log_loss_change=lambda t: t["cal_log_loss"] - t["raw_log_loss"])
        .round(4))

In [ ]:
cols = ["raw_brier", "cal_brier", "raw_log_loss", "cal_log_loss"]
runs = summary.assign(brier_change=summary["cal_brier"] - summary["raw_brier"],
                      log_loss_change=summary["cal_log_loss"] - summary["raw_log_loss"])


def _agg(d: pd.DataFrame) -> pd.Series:
    out = d[cols + ["brier_change", "log_loss_change"]].mean().round(4)
    for c in ("brier_change", "log_loss_change"):
        out[f"{c}_range"] = f"[{d[c].min():.4f}, {d[c].max():.4f}]"
    return out


avg = runs.groupby(["moment", "branch"], observed=True).apply(_agg)
avg.loc[("All", "All"), :] = _agg(runs)
avg

In [ ]:
from matplotlib.lines import Line2D

BRANCH_COLOR = {False: "C0", True: "C1"}      # same colors as plot_metric
BRANCH_OFFSET = {False: -0.18, True: 0.18}    # split each model row by branch
y_pos = {m: i for i, m in enumerate(MODEL_ORDER)}

fig, axes = plt.subplots(1, 2, figsize=FIG_FULL, sharey=True, sharex=True)
for ax, moment in zip(axes, MOMENT_ORDER):
    d = summary[summary["moment"] == moment]
    for balanced in (False, True):
        dd = d[d["balanced"] == balanced]
        y = dd["name"].map(y_pos).astype(float) + BRANCH_OFFSET[balanced]
        c = BRANCH_COLOR[balanced]
        ax.hlines(y, dd["raw_brier"], dd["cal_brier"], color=c, lw=1, alpha=0.6)
        ax.scatter(dd["raw_brier"], y, facecolors="white", edgecolors=c, s=22, zorder=3)
        ax.scatter(dd["cal_brier"], y, color=c, s=22, zorder=3)
    # no-skill reference: always predicting the true OFF rate
    ax.axvline(d["baseline_brier"].iloc[0], color="gray", ls=":", lw=1)
    ax.set_title(MOMENT_LABEL[moment])
    ax.set_xlabel("Brier score")

axes[0].set_yticks(range(len(MODEL_ORDER)))
axes[0].set_yticklabels([MODEL_LABEL[m] for m in MODEL_ORDER])
axes[0].invert_yaxis()

handles = [
    Line2D([], [], color=BRANCH_COLOR[False], marker="o", ls="", label=BRANCH_LABEL[False]),
    Line2D([], [], color=BRANCH_COLOR[True], marker="o", ls="", label=BRANCH_LABEL[True]),
    Line2D([], [], color="gray", marker="o", mfc="white", ls="", label="Raw"),
    Line2D([], [], color="gray", marker="o", ls="", label="Calibrated"),
    Line2D([], [], color="gray", ls=":", lw=1, label="No-skill baseline"),
]
fig.legend(handles=handles, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.08))
plt.tight_layout()
plt.show()

#### **OOF vs Holdout Brier**

In [ ]:
(summary.assign(brier_diff=summary["cal_brier"] - summary["oof_brier_true_scale"])
        .set_index(["moment", "model", "branch"])
        [["oof_brier_true_scale", "cal_brier", "brier_diff"]]
        .sort_values("brier_diff", key=abs, ascending=False)
        .round(4))

In [ ]:
summary["brier_diff"] = summary["cal_brier"] - summary["oof_brier_true_scale"]
plot_metric("brier_diff", "Holdout − OOF Brier score", ref=0, sharex=True)

## **Model Results**

#### **Model Discrimination**

In [ ]:
cols = {"raw_roc_auc": "ROC-AUC", "raw_pr_auc": "PR-AUC"}
for moment in MOMENT_ORDER:
    d = summary[summary["moment"] == moment]
    print(f"{MOMENT_LABEL[moment]} — ROC-AUC of a no-skill model: 0.500")
    print(f"{MOMENT_LABEL[moment]} — PR-AUC of a no-skill model: {d['true_off_rate'].iloc[0]:.3f}")
    display(d.set_index(["model", "branch"])[list(cols)]
             .rename(columns=cols)
             .sort_values("ROC-AUC", ascending=False)
             .round(3))

In [ ]:
# Print avg ROC-AUC and PR-AUC across moments
print("Average ROC-AUC and PR-AUC across moments:")
m1 = summary[summary["moment"] == "moment_1"]
m2 = summary[summary["moment"] == "moment_2"]

print(f"ROC-AUC: {m1['raw_roc_auc'].mean():.3f} (moment 1), {m2['raw_roc_auc'].mean():.5f} (moment 2)")
print(f"PR-AUC: {m1['raw_pr_auc'].mean():.3f} (moment 1), {m2['raw_pr_auc'].mean():.5f} (moment 2)")

In [ ]:
plot_metric("raw_roc_auc", "ROC-AUC")
plot_metric("raw_pr_auc", "PR-AUC")

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

# each model's better branch per moment, so every model appears once
best = (summary.sort_values("raw_roc_auc", ascending=False)
               .drop_duplicates(["moment", "name"])
               .sort_values(["moment", "name"]))

fig, axes = plt.subplots(2, 2, figsize=(TEXT_WIDTH_IN, TEXT_WIDTH_IN))
for j, moment in enumerate(MOMENT_ORDER):
    ax_roc, ax_pr = axes[0, j], axes[1, j]
    ax_roc.plot([0, 1], [0, 1], color="gray", ls=":", lw=1)
    for r in best[best["moment"] == moment].itertuples(index=False):
        g = preds[preds["run_id"] == r.run_id]
        fpr, tpr, _ = roc_curve(g["y"], g["p_raw"])
        precision, recall, _ = precision_recall_curve(g["y"], g["p_raw"])
        ax_roc.plot(fpr, tpr, color=MODEL_COLOR[r.name], lw=1, label=MODEL_LABEL[r.name])
        ax_pr.plot(recall, precision, color=MODEL_COLOR[r.name], lw=1)
    ax_pr.axhline(best.loc[best["moment"] == moment, "true_off_rate"].iloc[0], color="gray", ls=":", lw=1)
    ax_roc.set_title(MOMENT_LABEL[moment])
    ax_roc.set_xlabel("False positive rate")
    ax_roc.set_ylabel("True positive rate")
    ax_pr.set_xlabel("Recall")
    ax_pr.set_ylabel("Precision")
    ax_pr.set_ylim(0, 1)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()
plt.show()

#### **Moment Comparison**

In [ ]:
ID_COLS = ["GAME_ID", "EVENT_NUM"]
METRICS = {"raw_roc_auc": "ROC-AUC", "raw_pr_auc": "PR-AUC"}

keys = {m: set(map(tuple, preds.loc[preds["moment"] == m, ID_COLS].drop_duplicates().to_numpy()))
        for m in MOMENT_ORDER}
shared = keys["moment_1"] & keys["moment_2"]
print(f"shared events: {len(shared):,} | only moment_1: {len(keys['moment_1'] - shared)} "
      f"| only moment_2: {len(keys['moment_2'] - shared)}")

In [ ]:
wide = summary.pivot_table(index=["model", "branch"], columns="moment",
                           values=list(METRICS), observed=True)
moment_cmp = pd.concat({label: pd.DataFrame({
    MOMENT_LABEL["moment_1"]: wide[col]["moment_1"],
    MOMENT_LABEL["moment_2"]: wide[col]["moment_2"],
    "Change": wide[col]["moment_2"] - wide[col]["moment_1"],
}) for col, label in METRICS.items()}, axis=1)
moment_cmp.round(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(TEXT_WIDTH_IN, TEXT_WIDTH_IN * 0.75), sharex=True)
for i, balanced in enumerate([False, True]):
    d = summary[summary["balanced"] == balanced]
    for j, (col, label) in enumerate(METRICS.items()):
        ax = axes[i, j]
        for name, g in d.groupby("name", observed=True):
            g = g.sort_values("moment")
            ax.plot([0, 1], g[col], marker="o", ms=3, lw=1, color=MODEL_COLOR[name], label=MODEL_LABEL[name])
        ax.set_xticks([0, 1], [MOMENT_LABEL[m] for m in MOMENT_ORDER])
        ax.set_xlim(-0.2, 1.2)
        ax.set_title(f"{label} — {BRANCH_LABEL[balanced]}")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.07))
plt.tight_layout()
plt.show()

#### **Threshold behavior**

In [ ]:
cols = {"oof_threshold": "Threshold", "cal_f1_off": "F1"}
for moment in MOMENT_ORDER:
    print(MOMENT_LABEL[moment])
    display(summary[summary["moment"] == moment]
            .set_index(["model", "branch"])[list(cols)]
            .rename(columns=cols)
            .sort_values("F1", ascending=False)
            .round(3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(TEXT_WIDTH_IN, TEXT_WIDTH_IN * 0.8), sharex=True, sharey=True)
for i, moment in enumerate(MOMENT_ORDER):
    for j, balanced in enumerate([False, True]):
        ax = axes[i, j]
        s = sweep[(sweep["moment"] == moment) & (sweep["balanced"] == balanced)]
        for name, g in s.groupby("name", observed=True):
            ax.plot(g["threshold"], g["f1_off"], color=MODEL_COLOR[name], lw=1, label=MODEL_LABEL[name])
            saved = summary.loc[summary["run_id"] == g["run_id"].iloc[0]].iloc[0]
            ax.plot(saved["oof_threshold"], saved["cal_f1_off"], "o", ms=4, color=MODEL_COLOR[name])
        ax.set_title(f"{MOMENT_LABEL[moment]} — {BRANCH_LABEL[balanced]}")
        ax.set_xlim(0, 0.7)
for ax in axes[1]:
    ax.set_xlabel("Threshold")
for ax in axes[:, 0]:
    ax.set_ylabel("F1 (OFF)")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.07))
plt.tight_layout()
plt.show()

In [ ]:
thr = summary.assign(f1_lost=summary["holdout_best_f1"] - summary["cal_f1_off"])
print(pd.crosstab(thr["oof_threshold"], thr["holdout_best_threshold"],
                  rownames=["Saved (OOF)"], colnames=["Best on holdout"]))
(thr.set_index(["moment", "model", "branch"])
    [["oof_threshold", "holdout_best_threshold", "cal_f1_off", "holdout_best_f1", "f1_lost"]]
    .sort_values("f1_lost", ascending=False)
    .round(3))

#### **Threshold-Dependent Metrics**

In [ ]:
cols = {"oof_threshold": "Threshold", "cal_precision_off": "Precision", "cal_recall_off": "Recall",
        "cal_accuracy": "Accuracy", "cal_f1_off": "F1"}
for moment in MOMENT_ORDER:
    print(MOMENT_LABEL[moment])
    display(summary[summary["moment"] == moment]
            .set_index(["model", "branch"])[list(cols)]
            .rename(columns=cols)
            .sort_values("F1", ascending=False)
            .round(3))

In [ ]:
no_skill_acc = 1 - summary["true_off_rate"].iloc[0]   # accuracy of always predicting DEF
panels = [("cal_precision_off", "Precision (OFF)", None),
          ("cal_recall_off", "Recall (OFF)", None),
          ("cal_accuracy", "Accuracy", no_skill_acc),
          ("cal_f1_off", "F1 (OFF)", None)]

for moment in MOMENT_ORDER:
    d = summary[summary["moment"] == moment].sort_values("name")
    fig, axes = plt.subplots(2, 2, figsize=(TEXT_WIDTH_IN, TEXT_WIDTH_IN * 0.75), sharey=True)
    for ax, (col, label, ref) in zip(axes.flat, panels):
        for balanced in (False, True):
            dd = d[d["balanced"] == balanced]
            ax.scatter(dd[col], dd["model"], color=BRANCH_COLOR[balanced], label=BRANCH_LABEL[balanced])
        if ref is not None:
            ax.axvline(ref, color="gray", ls=":", lw=1)
        ax.set_xlabel(label)
    axes[0, 0].invert_yaxis()
    fig.suptitle(MOMENT_LABEL[moment], fontsize=10)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.05))
    plt.tight_layout()
    plt.show()

In [ ]:
metrics = {"cal_precision_off": "Precision", "cal_recall_off": "Recall",
           "cal_accuracy": "Accuracy", "cal_f1_off": "F1"}

rows = {}
for col, label in metrics.items():
    for moment in MOMENT_ORDER:
        d = summary[summary["moment"] == moment]
        lo, hi = d.loc[d[col].idxmin()], d.loc[d[col].idxmax()]
        rows[(label, MOMENT_LABEL[moment])] = {
            "Avg": round(d[col].mean(), 3),
            "Lowest": round(lo[col], 3),
            "Lowest model": f"{lo['model']} ({lo['branch']})",
            "Highest": round(hi[col], 3),
            "Highest model": f"{hi['model']} ({hi['branch']})",
        }

stats = pd.DataFrame.from_dict(rows, orient="index")
stats.index.names = ["Metric", "Moment"]
stats

## **Model selection**

In [ ]:
import os
from mlflow.tracking import MlflowClient
from sklearn.metrics import average_precision_score, roc_auc_score

Y_COL = "y_true"           # true label column
P_COL = "y_prob"           # raw model probability, BEFORE prior correction
TRUE_PRIOR_OFF = 0.232093  # same constant as calibrate.py

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"   # file-based store is in maintenance mode; allow reading it
client = MlflowClient(tracking_uri=(REPO_ROOT / "runs" / "mlruns").as_uri())


# OOF predictions: y (0/1), p_raw, and w, which reweights the rows to the real OFF rate
# (only matters for the Balanced runs, whose OOF rows are ~50/50).
def load_oof(run_id: str) -> pd.DataFrame:
    path = client.download_artifacts(run_id, "oof_predictions.csv")
    df = pd.read_csv(path).rename(columns={Y_COL: "y", P_COL: "p_raw"})[["y", "p_raw"]]
    q = df["y"].mean()
    df["w"] = np.where(df["y"] == 1, TRUE_PRIOR_OFF / q, (1 - TRUE_PRIOR_OFF) / (1 - q))
    return df

In [ ]:
def _oof_scores(run_id: str) -> pd.Series:
    d = load_oof(run_id)
    return pd.Series({"oof_roc_auc": roc_auc_score(d["y"], d["p_raw"], sample_weight=d["w"]),
                      "oof_pr_auc": average_precision_score(d["y"], d["p_raw"], sample_weight=d["w"])})


summary[["oof_roc_auc", "oof_pr_auc"]] = summary["run_id"].apply(_oof_scores).to_numpy()

(summary.set_index(["moment", "model", "branch"])
        [["oof_roc_auc", "raw_roc_auc", "oof_pr_auc", "raw_pr_auc"]]
        .rename(columns={"raw_roc_auc": "holdout_roc_auc", "raw_pr_auc": "holdout_pr_auc"})
        .round(3))

In [ ]:
### Selection on OOF, reporting on holdout:
### 1. For each moment, keep the runs within ROC_TOL of the best OOF ROC-AUC.
### 2. Among those, keep the runs within PR_TOL of the best OOF PR-AUC.
### 3. Among the remaining runs, pick the simplest model family (FAMILY_ORDER);
###    within the same family, the highest OOF PR-AUC wins.

ROC_TOL = 0.005   # OOF ROC-AUC differences smaller than this are treated as ties
PR_TOL = 0.005    # OOF PR-AUC differences smaller than this are treated as ties

best_roc = summary.groupby("moment", observed=True)["oof_roc_auc"].transform("max")
step1 = summary[summary["oof_roc_auc"] >= best_roc - ROC_TOL]
best_pr = step1.groupby("moment", observed=True)["oof_pr_auc"].transform("max")
candidates = (step1[step1["oof_pr_auc"] >= best_pr - PR_TOL]
              .assign(family=lambda t: t["name"].astype(str).map(MODEL_FAMILY),
                      family_rank=lambda t: t["family"].map(FAMILY_ORDER.index))
              .sort_values(["family_rank", "oof_pr_auc"], ascending=[True, False]))

for moment in MOMENT_ORDER:
    print(MOMENT_LABEL[moment])
    display(candidates[candidates["moment"] == moment]
            .set_index(["model", "branch"])[["family", "oof_roc_auc", "oof_pr_auc"]]
            .round(4))

chosen = (candidates.drop_duplicates("moment")
                    .set_index("moment")
                    .loc[MOMENT_ORDER])
print(chosen[["model", "branch", "run_id"]].to_string())

In [ ]:
cols = {"raw_roc_auc": "ROC-AUC", "raw_pr_auc": "PR-AUC", "cal_brier": "Brier",
        "cal_log_loss": "Log-loss", "oof_threshold": "Threshold",
        "cal_precision_off": "Precision", "cal_recall_off": "Recall",
        "cal_accuracy": "Accuracy", "cal_f1_off": "F1"}

(chosen.rename(index=MOMENT_LABEL)
       .set_index(["model", "branch"], append=True)[list(cols)]
       .rename(columns=cols)
       .round(3))

## **Extras** 

#### **Calibration Check**

The biggest gaps between mean predicted value and true offensive rate happen on the neural network models. 
This tests if the cause is that the calibrator was fitted to the fold models, but applied to a different model (the final one), and for the neural networks those two were not similar enough

In [ ]:
rows = []
for r in summary.itertuples(index=False):
    oof = load_oof(r.run_id)
    hold = preds[preds["run_id"] == r.run_id]
    row = {"moment": r.moment, "model": r.model, "branch": r.branch, "gap": r.gap}
    for label, df in (("oof", oof), ("holdout", hold)):
        row[f"{label}_def"] = df.loc[df["y"] == 0, "p_raw"].mean()
        row[f"{label}_off"] = df.loc[df["y"] == 1, "p_raw"].mean()
    rows.append(row)

shift = pd.DataFrame(rows)
p = summary["true_off_rate"].iloc[0]
shift["shift_def"] = shift["holdout_def"] - shift["oof_def"]
shift["shift_off"] = shift["holdout_off"] - shift["oof_off"]
shift["shift"] = p * shift["shift_off"] + (1 - p) * shift["shift_def"]   # weighted to the real OFF rate

(shift.set_index(["moment", "model", "branch"])
      [["oof_def", "holdout_def", "oof_off", "holdout_off", "shift", "gap"]]
      .sort_values("shift", key=abs, ascending=False)
      .round(4))

In [ ]:
groups = [
    ("Other models", ~shift["model"].isin([MODEL_LABEL["MLP"], MODEL_LABEL["HybridLSTM"]]), "#b0b0b0"),
    (MODEL_LABEL["MLP"], shift["model"] == MODEL_LABEL["MLP"], MODEL_COLOR["MLP"]),
    (MODEL_LABEL["HybridLSTM"], shift["model"] == MODEL_LABEL["HybridLSTM"], MODEL_COLOR["HybridLSTM"]),
]

fig, ax = plt.subplots(figsize=(TEXT_WIDTH_IN, TEXT_WIDTH_IN * 0.65))
ax.axhline(0, color="gray", ls=":", lw=1, zorder=1)
ax.axvline(0, color="gray", ls=":", lw=1, zorder=1)
for label, mask, color in groups:
    ax.scatter(shift.loc[mask, "shift"], shift.loc[mask, "gap"], s=50, color=color,
               edgecolor="white", linewidth=0.8, label=label, zorder=3)

# label the runs whose calibration gap exceeds 0.01:
# positive gap -> text to the left; leftmost point -> text above; others -> text to the right
flagged = shift[shift["gap"].abs() > 0.01]
for idx, r in flagged.iterrows():
    if r["gap"] > 0:
        pos = dict(xytext=(-8, 0), ha="right", va="center")
    elif idx == flagged["shift"].idxmin():
        pos = dict(xytext=(-3, 8), ha="left", va="bottom")
    else:
        pos = dict(xytext=(8, 0), ha="left", va="center")
    ax.annotate(f"{r['model']}\n{MOMENT_LABEL[r['moment']]}", (r["shift"], r["gap"]),
                textcoords="offset points", fontsize=9, linespacing=1.1, **pos)

ax.set_xlabel("Raw score shift (holdout − OOF)")
ax.set_ylabel("Calibration gap")
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.07))
plt.tight_layout()
plt.show()

#### **Detection Confidence**

ROC-AUC and lift (PR-AUC / OFF rate of the tier, 1 = no skill) on the holdout, split by the shot and rim detection confidence tiers. Lift is used instead of PR-AUC because the tiers have different OFF rates.

In [ ]:
DETECTORS = {"Shot": "SHOT_DETECTION_CONFIDENCE", "Rim": "RIM_DETECTION_CONFIDENCE"}
TIER_ORDER = ["All", "HIGH", "MEDIUM", "LOW", "FALLBACK"]

# both moments score the same events, and moment_2's holdout file holds both tiers
conf = pd.read_csv(REPO_ROOT / "data" / "sequences" / "moment_2" / "features" / "tabular" / "sequences_fe_tabular_holdout.csv",
                   usecols=ID_COLS + list(DETECTORS.values()))
merged = preds.merge(conf, on=ID_COLS, how="left", validate="many_to_one")
tiers = pd.concat([merged.assign(detector=d, tier=merged[c]) for d, c in DETECTORS.items()]
                  + [merged.assign(detector=d, tier="All") for d in DETECTORS], ignore_index=True)
assert tiers["tier"].notna().all(), "holdout predictions without a detection tier"
tiers["detector"] = pd.Categorical(tiers["detector"], categories=list(DETECTORS), ordered=True)
tiers["tier"] = pd.Categorical(tiers["tier"], categories=TIER_ORDER, ordered=True)


def _disc(g: pd.DataFrame) -> pd.Series:
    pr_auc = average_precision_score(g["y"], g["p_raw"])
    return pd.Series({"n": len(g), "roc_auc": roc_auc_score(g["y"], g["p_raw"]), "lift": pr_auc / g["y"].mean()})


tier_disc = (tiers.groupby(["detector", "tier", "moment", "branch", "name"], observed=True)[["y", "p_raw"]]
                  .apply(_disc)
                  .reset_index())
tier_disc["n"] = tier_disc["n"].astype(int)

In [ ]:
# average of the eight models, per moment and branch
(tier_disc.groupby(["detector", "tier", "n", "moment", "branch"], observed=True)[["roc_auc", "lift"]]
          .mean()
          .unstack(["moment", "branch"])
          .rename(columns=MOMENT_LABEL, level="moment")
          .round(3))

In [ ]:
# selected model of each moment (see Model selection)
for moment, r in chosen.iterrows():
    print(f"{MOMENT_LABEL[moment]}: {r['model']} ({r['branch']})")

(tier_disc.merge(chosen.reset_index()[["moment", "name", "branch"]], on=["moment", "name", "branch"])
          .pivot_table(index=["detector", "tier", "n"], columns="moment", values=["roc_auc", "lift"], observed=True)
          [["roc_auc", "lift"]]
          .rename(columns=MOMENT_LABEL, level="moment")
          .round(3))